# Setup

In [ ]:
# Initialize notebook
import json
import plotly.graph_objects as go
import numpy as np
import pandas as pd
import sys
sys.path.append('../../assets/python/')
import tfb
import dmg5e
import estats5e


METADATA = {'Contributor': 'T. Dunn'}
SAVEFIGS = False

PC_CLASSES = {
    'Barbarian': {'color': '#E7623E', 'group': 'martial'},
    'Bard': {'color': '#AB6DAC', 'group': 'spellcaster'},
    'Cleric': {'color': '#91A1B2', 'group': 'spellcaster'},
    'Druid': {'color': '#7A853B', 'group': 'spellcaster'},
    'Fighter': {'color': '#7F513E', 'group': 'martial'},
    'Monk': {'color': '#51A5C5', 'group': 'martial'},
    'Paladin': {'color': '#B59E54', 'group': 'martial-spellcaster'},
    'Ranger': {'color': '#507F62', 'group': 'martial-spellcaster'},
    'Rogue': {'color': '#555752', 'group': 'martial'},
    'Sorcerer': {'color': '#992E2E', 'group': 'spellcaster'},
    'Warlock': {'color': '#7B469B', 'group': 'spellcaster'},
    'Wizard': {'color': '#2A50A1', 'group': 'spellcaster'}
}

ENCOUNTER_DIFFICULTIES = {
    'Easy':   {
        'XP': [dmg5e.pc_xp_thresholds(lvl)['Easy'] for lvl in range(1,21)], 
        'color': '#1F77B4'
    },
    'Medium': {
        'XP': [dmg5e.pc_xp_thresholds(lvl)['Medium'] for lvl in range(1,21)], 
        'color': '#FF7F0E'
    },
    'Hard':   {
        'XP': [dmg5e.pc_xp_thresholds(lvl)['Hard'] for lvl in range(1,21)], 
        'color': '#2CA02C'
    },
    'Deadly': {
        'XP': [dmg5e.pc_xp_thresholds(lvl)['Deadly'] for lvl in range(1,21)], 
        'color': '#D62728'
    },
    'Very Deadly':  {
        'XP': [0.5*dmg5e.pc_xp_thresholds(lvl)['Daily'] for lvl in range(1,21)], 
        'color': '#9467BD'
    },
    'Daily':  {
        'XP': [dmg5e.pc_xp_thresholds(lvl)['Daily'] for lvl in range(1,21)], 
        'color': '#8C564B'
    },
}

COLORS = [
    '#1f77b4',  # muted blue
    '#ff7f0e',  # safety orange
    '#2ca02c',  # cooked asparagus green
    '#d62728',  # brick red
    '#9467bd',  # muted purple
    '#8c564b',  # chestnut brown
    '#e377c2',  # raspberry yogurt pink
    '#7f7f7f',  # middle gray
    '#bcbd22',  # curry yellow-green
    '#17becf',  # blue-teal
    '#1f77b4',  # muted blue
    '#ff7f0e',  # safety orange
    '#2ca02c',  # cooked asparagus green
    '#d62728',  # brick red
    '#9467bd',  # muted purple
    '#8c564b',  # chestnut brown
    '#e377c2',  # raspberry yogurt pink
    '#7f7f7f',  # middle gray
    '#bcbd22',  # curry yellow-green
    '#17becf',  # blue-teal
]

In [56]:
# useful functions
import pandas as pd

def attack_hit_crit_prob(AC, AB):
    return max(0.05, min(0.95, 0.05*(21 + AB - AC)))
    #return 0.05*(21 + AB - AC)

def calc_damage_ratio(dfP, dfM, lvls, cr):

    p_ac  = [dfP['adj AC'][lvl]  for lvl in lvls]
    p_ab  = [dfP['adj AB'][lvl]  for lvl in lvls]
    p_dpr = [dfP['adj DPR'][lvl] for lvl in lvls]
    p_hp  = [dfP['adj HP'][lvl]  for lvl in lvls]

    m_ac  = dfM['adj AC'][cr]
    m_ab  = dfM['adj AB'][cr]
    m_dpr = dfM['adj DPR'][cr]
    m_hp  = dfM['adj HP'][cr]

    rtw = m_hp/sum([dpr*attack_hit_crit_prob(m_ac, ab) for ab, dpr in zip(p_ab,p_dpr)])
    dmg = rtw*m_dpr*np.mean([attack_hit_crit_prob(ac, m_ab) for ac in p_ac])
    return dmg/sum(p_hp)

def calc_xp_ratio(dfP, dfM, lvls, cr):
    p_xp  = [dfP['Encounter XP'][lvl] for lvl in lvls]
    m_xp  = dfM['XP'][cr]
    return m_xp/sum(p_xp)

def calc_exp_ratio(dfP, dfM, lvls, cr):
    p_xp  = [dfP['Encounter eXP'][lvl] for lvl in lvls]
    m_xp  = dfM['eXP'][cr]
    return m_xp/sum(p_xp)

def load_pc_data(file, columns):
    """Loads PC data from file and puts it into a pandas dataframe
    """
    with open(file, 'r') as fin:
        pc_data = json.load(fin)

    pc_dict = {}
    for c in columns:
        d = []
        for pc in pc_data:
            d += [x[c] for x in pc_data[pc]]
        pc_dict[c] = d

    return pd.DataFrame(pc_dict)

# Figures

In [48]:
# Fig. 1: Plots monster XP / (HP x DPR) vs AC + AB

COEFF = 0.25 #(0.65/3)

# create figure
fig = go.Figure()
colors = iter(COLORS)

# plot DMG data
book = 'DMG'
dfM = pd.read_csv('../../assets/data/dmg-targets.csv')
dfM['adj AB + AC'] = dfM['AB'] + dfM['AC']
dfM['XP ratio'] = dfM['XP']/(COEFF*dfM['DPR Mean']*dfM['HP Mean'])
dfM = dfM[dfM['CR'].between(1, 30)]
dfM = dfM[['CR','adj AB + AC','XP ratio']].groupby('CR').mean().reset_index()
dfM['text'] = dfM.apply(lambda row: 
    f'<b>{book}</b><br>'+
    f"CR: {row['CR']:.0f}<br>"+
    f"AC + AB: {row['adj AB + AC']:.0f}<br>"+
    f"XP ratio: {row['XP ratio']:.2f}", axis=1)

fig.add_scatter(
    x=dfM['adj AB + AC'].tolist(), 
    y=dfM['XP ratio'].tolist(), 
    mode='markers',
    name=book, 
    legendgroup=book, 
    line_color=next(colors),
    hovertext=dfM['text'],
    hoverinfo='text',
)

# plot official monsters
book = '5e (2014) monsters'
dfM = pd.read_csv('../../assets/data/monsters-2014.csv')
dfM = dfM[dfM['CR'].between(1, 30)]
dfM['adj AB + AC'] = dfM['adj AB'] + dfM['adj AC']
dfM['XP ratio'] = dfM['XP']/(COEFF*dfM['adj DPR']*dfM['adj HP'])
dfM = dfM[['CR','adj AB + AC','XP ratio']].groupby('CR').mean().reset_index()
dfM['text'] = dfM.apply(lambda row: 
    f'<b>{book}</b><br>'+
    f"CR: {row['CR']:.0f}<br>"+
    f"AC + AB: {row['adj AB + AC']:.0f}<br>"+
    f"XP ratio: {row['XP ratio']:.2f}", axis=1)

fig.add_scatter(
    x=dfM['adj AB + AC'].tolist(), 
    y=dfM['XP ratio'].tolist(), 
    mode='markers',
    name=book, 
    legendgroup=book, 
    line_color=next(colors),
    hovertext=dfM['text'],
    hoverinfo='text',
)

book = 'linear approximation'
x = np.linspace(0, 45, 100)
fig.add_trace(go.Scatter(
    x=x, 
    y=(1 + (x - 15)/13),
    mode='lines',
    name=f'{book}',
    legendgroup=book,
    showlegend=True,
    line_color='black',
    line_dash='dash',
    hoverinfo='skip',
))

# update layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='AC + AB', range=[15, 45], dtick=5, minor_dtick=1),
    yaxis=dict(title_text='XP / (HP x DPR)', range=[0,4], tickformat='.1f', dtick=0.5, minor_dtick=0.1),
    legend=dict(xanchor='right', yanchor='bottom', x=1.00, y=0.00),
    width=600, 
    height=500,
)

# show figure
fig.show(config=tfb.FIG_CONFIG)

# save figures
if SAVEFIGS:
    fig.write_image('../../assets/images/xp-approximations.png')
    fig.update_layout(autosize=True, width=None, height=None)
    tfb.save_fig_html(fig, format='large', name=f'./fig-npc-accuracy-vs-acab-large')
    tfb.save_fig_html(fig, format='small', name=f'./fig-npc-accuracy-vs-acab-small', marker_size=4)

In [49]:
# Fig. 2: Plot PC XP/(DPR x HP) vs AB + AC

# load PC data
df = load_pc_data(
    f'../../_classes/xp-and-player-characters/pc data - Medium 2 short rests.json',
    ['level','armor class','attack bonus','hit points mean','hit points multiplier','damage per round mean'],
)
df['adj hit points mean'] = df['hit points mean']*df['hit points multiplier']
df['AC + AB'] = df['armor class'] + df['attack bonus']
df = df.groupby(['level']).mean().reset_index()

# create figure
fig = go.Figure()

COEFF = 1.0 # 4*0.65/3
groups = [
    #{'name': 'Easy'       , 'XP': np.array(ENCOUNTER_DIFFICULTIES['Easy']['XP'])       , 'color': ENCOUNTER_DIFFICULTIES['Easy']['color']},
    {'name': 'Medium'     , 'XP': np.array(ENCOUNTER_DIFFICULTIES['Medium']['XP'])     , 'color': ENCOUNTER_DIFFICULTIES['Medium']['color']},
    {'name': 'Hard'       , 'XP': np.array(ENCOUNTER_DIFFICULTIES['Hard']['XP'])       , 'color': ENCOUNTER_DIFFICULTIES['Hard']['color']},
    {'name': 'Deadly'     , 'XP': np.array(ENCOUNTER_DIFFICULTIES['Deadly']['XP'])     , 'color': ENCOUNTER_DIFFICULTIES['Deadly']['color']},
    {'name': 'Very Deadly', 'XP': np.array(ENCOUNTER_DIFFICULTIES['Very Deadly']['XP']), 'color': ENCOUNTER_DIFFICULTIES['Very Deadly']['color']},
]
for group in groups:
    name = group['name']

    df['XP ratio'] = group['XP']/(COEFF*df['adj hit points mean']*df['damage per round mean'])
    df['text'] = df.apply(lambda row: 
            f"<b>{name}</b><br>"+
            f"level: {row['level']:.0f}<br>"+
            f"AC + AB: {row['AC + AB']:.0f}<br>"+
            f"XP ratio: {row['XP ratio']:.2f}", axis=1)
    
    color = group['color']
    fig.add_trace(go.Scatter(
        x=df['AC + AB'], 
        y=df['XP ratio'],
        mode='markers', 
        line=dict(color=color, dash='solid'),
        name=name,
        legendgroup=name,
        hovertext=df['text'],
        hoverinfo='text',
    ))
    fig.add_trace(go.Scatter(
        x=[10,45], 
        y=2*[df['XP ratio'].mean()],
        mode='lines', 
        line=dict(color=color, dash='dash'),
        name=name,
        legendgroup=name,
        showlegend=False,
        hoverinfo='skip',
    ))

x = np.linspace(10,45,10)
fig.add_trace(go.Scatter(
    x=x, 
    y=(1 + (x - 15)/13),
    mode='lines', 
    line=dict(color='black', dash='dash'),
    name='linear approximation',
    showlegend=False,
    hoverinfo='skip',
))

# update layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='AC + AB', range=[15, 45], dtick=5, minor_dtick=1),
    yaxis=dict(title_text='XP / (HP x DPR)', range=[0,4], tickformat='.1f', dtick=0.5, minor_dtick=0.1),
    legend=dict(yanchor='top',  y=1.00, xanchor='left', x=0.00),
    width=600, 
    height=500,
)

# show figure
fig.show(config=tfb.FIG_CONFIG)

# save figures
if SAVEFIGS:
    fig.update_layout(autosize=True, width=None, height=None)
    tfb.save_fig_html(fig, format='large', name=f'./fig-pc-accuracy-vs-acab-large')
    tfb.save_fig_html(fig, format='small', name=f'./fig-pc-accuracy-vs-acab-small', marker_size=4, legend_font_size=8)

In [50]:
# Fig. 3: plots XP accuracy vs level for encounter with a single CR appropriate monster
import pandas as pd

xp_method = 'linear'

# pc stats
classes = list(PC_CLASSES.keys())
pc_data_file = f'../../_classes/xp-and-player-characters/pc data - Medium 2 short rests.json'
dfP = load_pc_data(pc_data_file, [
    'class','level',
    'hit points mean',
    'hit points multiplier',
    'damage per round mean',
    'attack bonus','armor class',
    'encounter XP mean'
])
dfP['adj hit points mean'] = dfP['hit points mean']*dfP['hit points multiplier']
dfP = dfP.rename({
    'adj hit points mean': 'adj HP', 
    'damage per round mean': 'adj DPR', 
    'attack bonus': 'adj AB', 
    'armor class': 'adj AC',
    'encounter XP mean': 'eXP',
}, axis=1)
xp_values = 0.5*np.array(ENCOUNTER_DIFFICULTIES['Daily']['XP'])
dfP['Encounter XP'] = dfP.apply(lambda row: xp_values[row['level']-1], axis=1)
dfP['Encounter eXP']  = dfP.apply(lambda row: estats5e.effXP(row['adj HP'], row['adj AC'], row['adj DPR'], row['adj AB'], method=xp_method, ctype='PC'), axis=1)
dfP = dfP[dfP['class'].isin(classes)]
dfP = dfP[['level','adj HP','adj DPR','adj AC','adj AB','Encounter XP','Encounter eXP']].groupby(['level']).mean()

# monster stats
dfM = pd.read_csv('../../assets/data/monsters-2014.csv')
dfM['adj AC']  = 0.5*(dfM['adj AC'] + dfM['adj SB'] + 14)
dfM['eXP']  = dfM.apply(lambda row: estats5e.effXP(row['adj HP'], row['adj AC'], row['adj DPR'], row['adj AB'], method=xp_method, ctype='NPC'), axis=1)
dfM = dfM[dfM['CR'].between(1, 28)]
dfM.sort_values(by='CR', inplace=True)
dfM = dfM[['CR','HP','adj HP','AC','adj AC','adj DPR','adj AB','XP','eXP']].groupby(['CR']).mean()

# create figure
fig = go.Figure()
colors = iter(COLORS)
smooth_factor = 1

name = '5e official'
cr_offset = 0
crs = dfM.index.to_list()
dmg_ratio = np.array([calc_damage_ratio(dfP, dfM, 4*[max(1,min(cr+cr_offset,20))], cr) for cr in crs])
xp_ratio = np.array([calc_xp_ratio(dfP, dfM, 4*[max(1,min(cr+cr_offset,20))], cr) for cr in crs])
fig = tfb.plot_data_and_spline(fig, 
    x=crs, 
    y=xp_ratio/dmg_ratio,
    smooth_factor=smooth_factor, 
    name=name,
    legendgroup=name,
    line_color=next(colors),
    hovertemplate=
            f'<b>{name}</b><br>'+
            'challenge rating %{x}<br>'+
            'accuracy %{y:.2f}'+
            '<extra></extra>',
)

name = 'XP - linear'
dmg_ratio = np.array([calc_damage_ratio(dfP, dfM, 4*[max(1,min(cr+cr_offset,20))], cr) for cr in crs])
xp_ratio = np.array([calc_exp_ratio(dfP, dfM, 4*[max(1,min(cr+cr_offset,20))], cr) for cr in crs])
fig = tfb.plot_data_and_spline(fig, 
    x=crs, 
    y=xp_ratio/dmg_ratio,
    smooth_factor=smooth_factor, 
    name=name,
    legendgroup=name,
    line_color=next(colors),
    hovertemplate=
            f'<b>{name}</b><br>'+
            'challenge rating %{x}<br>'+
            'accuracy %{y:.2f}'+
            '<extra></extra>',
)

name = 'XP - exponential'
xp_method = 'exp'
dfP['Encounter eXP']  = dfP.apply(lambda row: estats5e.effXP(row['adj HP'], row['adj AC'], row['adj DPR'], row['adj AB'], method=xp_method, ctype='PC'), axis=1)
dfM['eXP']  = dfM.apply(lambda row: estats5e.effXP(row['adj HP'], row['adj AC'], row['adj DPR'], row['adj AB'], method=xp_method, ctype='NPC'), axis=1)
dmg_ratio = np.array([calc_damage_ratio(dfP, dfM, 4*[max(1,min(cr+cr_offset,20))], cr) for cr in crs])
xp_ratio = np.array([calc_exp_ratio(dfP, dfM, 4*[max(1,min(cr+cr_offset,20))], cr) for cr in crs])
fig = tfb.plot_data_and_spline(fig, 
    x=crs, 
    y=xp_ratio/dmg_ratio,
    smooth_factor=smooth_factor, 
    name=name,
    legendgroup=name,
    line_color=next(colors),
    hovertemplate=
            f'<b>{name}</b><br>'+
            'challenge rating %{x}<br>'+
            'accuracy %{y:.2f}'+
            '<extra></extra>',
)

fig.add_scatter(
    x=[1,30],
    y=[1,1],
    mode='lines',
    showlegend=False,
    line_color='black',
    line_dash='dash',
)

# show figure
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='challenge rating', range=[0.8, 30.2], dtick=5, minor_dtick=1),
    yaxis=dict(title_text='accuracy', range=[0, 1.6], dtick=0.2, tickformat='.1f', minor_dtick=0.1),
    legend=dict(xanchor='left', x=0.01, yanchor='bottom', y=0.01),
    width=600, 
    height=500,
)
fig.show(config=tfb.FIG_CONFIG)

# save figures
if SAVEFIGS:
    fig.update_layout(autosize=True, width=None, height=None)
    tfb.save_fig_html(fig, format='large', name=f'./fig-xp-accuracy-vs-cr-matched-large')
    tfb.save_fig_html(fig, format='small', name=f'./fig-xp-accuracy-vs-cr-matched-small', marker_size=4)

In [63]:
# Fig. 4: plots XP accuracy vs level for encounter with a single monster
import pandas as pd

# pc stats
pc_data_file = f'../../_classes/xp-and-player-characters/pc data - Medium 2 short rests.json'
dfP = load_pc_data(pc_data_file, ['class','level','hit points mean','hit points multiplier','damage per round mean','attack bonus','armor class'])
dfP['adj hit points mean'] = dfP['hit points mean']*dfP['hit points multiplier']
dfP = dfP.rename({'adj hit points mean': 'adj HP', 'damage per round mean': 'adj DPR', 'attack bonus': 'adj AB', 'armor class': 'adj AC'}, axis=1)
dfP['Encounter XP'] = dfP.apply(lambda row: ENCOUNTER_DIFFICULTIES['Very Deadly']['XP'][row['level']-1], axis=1)
dfP = dfP[['level','adj HP','adj DPR','adj AC','adj AB','Encounter XP']].groupby(['level']).mean()

# monster stats
dfM = pd.read_csv('../../assets/data/monsters-2014.csv')
dfM['adj AC']  = 0.5*(dfM['adj AC'] + dfM['adj SB'] + 14)
dfM = dfM[dfM['CR'].between(1, 28)]
dfM.sort_values(by='CR', inplace=True)
dfM = dfM[['CR','HP','adj HP','AC','adj AC','adj DPR','adj AB','XP']].groupby(['CR']).mean()

# create figure
fig = go.Figure()

colors = iter(COLORS)

########################################################################################
name = f'5e official'
levels = dfP.index.tolist()
crs = dfM.index.tolist()
data = {'rel CR': [], 'accuracy': [], 'text': []}
for lvl in levels:
    for cr in crs:
        accuracy = calc_xp_ratio(dfP, dfM, 4*[lvl], cr)/calc_damage_ratio(dfP, dfM, 4*[lvl], cr)
        data['rel CR'].append(cr - lvl)
        data['accuracy'].append(accuracy)
        data['text'].append(
            f'<b>{name}</b><br>'+
            f'accuracy: {accuracy:.2f}<br>'+
            f'CR - level: {cr - lvl:.0f}<br>'+
            f'CR: {cr:.0f}<br>'+
            f'level: {lvl:.0f}')

data['rel CR'] = tfb.jitter(data['rel CR'], 0.01)
dfA = pd.DataFrame(data)
dfA = dfA[dfA['accuracy'].gt(0)]
dfA.sort_values(by='rel CR', inplace=True)

smooth_factor = 1.e8
fig = tfb.plot_data_and_spline(fig, 
    x=dfA['rel CR'], 
    y=dfA['accuracy'],
    smooth_factor=smooth_factor, 
    name=name,
    legendgroup=name,
    line=dict(color=next(colors), dash='solid', width=2),
    marker_size=3,
    hovertext=dfA['text'],
    hoverinfo='text',
)

########################################################################################
name = f'XP - linear'
xp_method = 'linear'
dfP['Encounter eXP']  = dfP.apply(lambda row: estats5e.effXP(row['adj HP'], row['adj AC'], row['adj DPR'], row['adj AB'], method=xp_method, ctype='PC'), axis=1)
dfM['eXP']  = dfM.apply(lambda row: estats5e.effXP(row['adj HP'], row['adj AC'], row['adj DPR'], row['adj AB'], method=xp_method, ctype='NPC'), axis=1)

levels = dfP.index.tolist()
crs = dfM.index.tolist()
data = {'rel CR': [], 'accuracy': [], 'text': []}
for lvl in levels:
    for cr in crs:
        accuracy = calc_exp_ratio(dfP, dfM, 4*[lvl], cr)/calc_damage_ratio(dfP, dfM, 4*[lvl], cr)
        data['rel CR'].append(cr - lvl)
        data['accuracy'].append(accuracy)
        data['text'].append(
            f'<b>{name}</b><br>'+
            f'accuracy: {accuracy:.2f}<br>'+
            f'CR - level: {cr - lvl:.0f}<br>'+
            f'CR: {cr:.0f}<br>'+
            f'level: {lvl:.0f}')

data['rel CR'] = tfb.jitter(data['rel CR'], 0.01)
dfA = pd.DataFrame(data)
dfA = dfA[dfA['accuracy'].gt(0)]
dfA.sort_values(by='rel CR', inplace=True)

smooth_factor = 1.e8
fig = tfb.plot_data_and_spline(fig, 
    x=dfA['rel CR'], 
    y=dfA['accuracy'],
    smooth_factor=smooth_factor, 
    name=name,
    legendgroup=name,
    line=dict(color=next(colors), dash='solid', width=2),
    marker_size=3,
    hovertext=dfA['text'],
    hoverinfo='text',
)

########################################################################################
name = f'XP - exponential'
xp_method = 'exp'
dfP['Encounter eXP']  = dfP.apply(lambda row: estats5e.effXP(row['adj HP'], row['adj AC'], row['adj DPR'], row['adj AB'], method=xp_method, ctype='PC'), axis=1)
dfM['eXP']  = dfM.apply(lambda row: estats5e.effXP(row['adj HP'], row['adj AC'], row['adj DPR'], row['adj AB'], method=xp_method, ctype='NPC'), axis=1)

levels = dfP.index.tolist()
crs = dfM.index.tolist()
data = {'rel CR': [], 'accuracy': [], 'text': []}
for lvl in levels:
    for cr in crs:
        accuracy = calc_exp_ratio(dfP, dfM, 4*[lvl], cr)/calc_damage_ratio(dfP, dfM, 4*[lvl], cr)
        data['rel CR'].append(cr - lvl)
        data['accuracy'].append(accuracy)
        data['text'].append(
            f'<b>{name}</b><br>'+
            f'accuracy: {accuracy:.2f}<br>'+
            f'CR - level: {cr - lvl:.0f}<br>'+
            f'CR: {cr:.0f}<br>'+
            f'level: {lvl:.0f}')

data['rel CR'] = tfb.jitter(data['rel CR'], 0.01)
dfA = pd.DataFrame(data)
dfA = dfA[dfA['accuracy'].gt(0)]
dfA.sort_values(by='rel CR', inplace=True)

smooth_factor = 1.e8
fig = tfb.plot_data_and_spline(fig, 
    x=dfA['rel CR'], 
    y=dfA['accuracy'],
    smooth_factor=smooth_factor, 
    name=name,
    legendgroup=name,
    line=dict(color=next(colors), dash='solid', width=2),
    marker_size=3,
    hovertext=dfA['text'],
    hoverinfo='text',
)

# reference line
fig.add_scatter(
    x=[-30,30], 
    y=[1,1],
    mode='lines',
    showlegend=False,
    line=dict(color='black', dash='dash'),
    hoverinfo='skip',
)

# show figure
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='challenge rating - party level', range=[-20.8,30.8], dtick=5, minor_dtick=1),
    yaxis=dict(title_text='accuracy', range=[0,2.5], dtick=0.5, tickformat='.1f', minor_dtick=0.1),
    legend=dict(xanchor='left', x=0.01, yanchor='bottom', y=0.01),
    width=600,
    height=500,
)
fig.show(config=tfb.FIG_CONFIG)

# save figures
if SAVEFIGS:
    fig.update_layout(autosize=True, width=None, height=None)
    tfb.save_fig_html(fig, format='large', name=f'./fig-xp-accuracy-vs-relative-cr-large', marker_size=3)
    tfb.save_fig_html(fig, format='small', name=f'./fig-xp-accuracy-vs-relative-cr-small', marker_size=3)

In [57]:
# Fig. 5: Plots the effective encounter multiplier vs the number of monsters for official 5e XP

# pc stats
classes = list(PC_CLASSES.keys())
pc_data_file = f'../../_classes/xp-and-player-characters/pc data - Medium 2 short rests.json'
dfP = load_pc_data(pc_data_file, [
    'class','level',
    'hit points mean',
    'hit points multiplier',
    'damage per round mean',
    'attack bonus','armor class',
    'encounter XP mean'
])
dfP['adj hit points mean'] = dfP['hit points mean']*dfP['hit points multiplier']
dfP = dfP.rename({
    'adj hit points mean': 'adj HP', 
    'damage per round mean': 'adj DPR', 
    'attack bonus': 'adj AB', 
    'armor class': 'adj AC',
    'encounter XP mean': 'Encounter eXP',
}, axis=1)
for k in ENCOUNTER_DIFFICULTIES:
    xp_values = np.array(ENCOUNTER_DIFFICULTIES[k]['XP'])
    dfP[f'{k} XP'] = dfP.apply(lambda row: xp_values[row['level']-1], axis=1)
dfP[f'Encounter XP'] = 0.5*dfP[f'Daily XP']
dfP = dfP[dfP['class'].isin(classes)]
dfP = dfP[[
    'level','adj HP','adj DPR','adj AC','adj AB',
    'Encounter eXP','Easy XP','Medium XP','Hard XP','Deadly XP','Encounter XP','Daily XP']].groupby(['level']).mean()

# monster stats
dfM = pd.read_csv('../../assets/data/monsters-2014.csv')
dfM['adj AC']  = 0.5*(dfM['adj AC'] + dfM['adj SB'] + 14)
dfM = dfM[dfM['CR'].between(1, 28)]
dfM.sort_values(by='CR', inplace=True)
dfM = dfM[['CR','HP','adj HP','AC','adj AC','adj DPR','adj AB','XP']].groupby(['CR']).mean()

# create figure
fig = go.Figure()


#difficulties = ['Easy','Medium','Hard','Deadly']
difficulties = {
    'Easy':   ['Easy'  ,'Medium'     ],
    'Medium': ['Medium','Hard'       ],
    'Hard':   ['Hard'  ,'Deadly'     ],
    'Deadly': ['Deadly','Very Deadly'],
}
party_size = 4
levels = dfP.index.tolist()
crs = dfM.index.tolist()
colors = iter(COLORS)

data = {
    'difficulty': [],
    'PC level': [],
    'PC number': [],
    'PC XP min': [],
    'PC XP max': [],
    'monster CR': [],
    'monster number': [],
    'monster XP': [],
    'damage ratio': [],
    'XP ratio': [],
    'accuracy': [],
    'text': []
}
for difficulty in difficulties:
    for level in levels:
        for cr in crs:
            diff_min = difficulties[difficulty][0]
            diff_max = difficulties[difficulty][1]
            xp_min = ENCOUNTER_DIFFICULTIES[diff_min]['XP'][level-1]
            xp_max = ENCOUNTER_DIFFICULTIES[diff_max]['XP'][level-1]
            monster_xp = dfM['XP'][cr]
            n_monsters = np.round(party_size*0.5*(xp_max + xp_min)/monster_xp, 0)
            if n_monsters*monster_xp < party_size*xp_min: n_monsters += 1
            if n_monsters*monster_xp >= party_size*xp_max: n_monsters -= 1
            encounter_xp = n_monsters*monster_xp
            if encounter_xp >= party_size*xp_min and encounter_xp <= party_size*xp_max:
                data['difficulty'].append(difficulty)
                data['PC level'].append(level)
                data['PC number'].append(party_size)
                data['PC XP min'].append(xp_min)
                data['PC XP max'].append(xp_max)
                data['monster CR'].append(cr)
                data['monster XP'].append(monster_xp)
                data['damage ratio'].append(calc_damage_ratio(dfP, dfM, party_size*[level], cr))
                data['XP ratio'].append(calc_xp_ratio(dfP, dfM, party_size*[level], cr))
                data['monster number'].append(n_monsters)
                eff_multiplier = data['XP ratio'][-1]/data['damage ratio'][-1]
                data['accuracy'].append(eff_multiplier)
                data['text'].append(
                    f'<b>{difficulty}</b><br>'+
                    f'PCs: {party_size:.0f} @ level {level:.0f}<br>'+
                    f'NPCs: {n_monsters:.0f} @ CR {cr:.0f}<br>'+
                    f'effective EM: {eff_multiplier:.2f}'
                )

data['monster number'] = tfb.jitter(data['monster number'], 0.01)
dfA = pd.DataFrame(data)

for difficulty in difficulties:
    dfD = dfA[dfA['difficulty'].eq(difficulty) & dfA['monster number'].between(0.5, 30)]
    dfD = dfD.sort_values(by='monster number')

    fig = tfb.plot_data_and_spline(fig, 
        x=dfD['monster number'], 
        y=dfD['accuracy'],
        smooth_factor=1e3, 
        name=difficulty,
        legendgroup=difficulty,
        line=dict(color=next(colors), dash='solid', width=2),
        marker_size=3,
        hovertext=dfD['text'],
        hoverinfo='text',
    )

n_list = list(range(1,21))
y_list = [dmg5e.encounter_xp_multiplier(party_size, n) for n in n_list]
fig.add_scatter(
    x=n_list, 
    y=y_list,
    mode='markers+lines',
    name=f'encounter multiplier',
    line_color='black',
    hovertemplate=
        '<b>encounter multiplier</b><br>'+
        f'PCs: {party_size:.0f}<br>'+
        'NPCs: %{x:.0f}<br>'+
        'EM: %{y:.2f}'+
        '<extra></extra>'
)

# show figure
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='monsters', range=[-0.2, 20.2], dtick=5, minor_dtick=1),
    yaxis=dict(title_text='effective multiplier', range=[0, 5], dtick=1, tickformat='.0f', minor_dtick=0.5),
    legend=dict(xanchor='left', x=0.01, yanchor='top',  y=0.99, bgcolor='rgba(0,0,0,0)'),
    width=600, 
    height=500,
)
fig.show(config=tfb.FIG_CONFIG)

# save figures
if SAVEFIGS:
    fig.update_layout(autosize=True, width=None, height=None)
    tfb.save_fig_html(fig, format='large', name=f'./fig-effective-xp-multiplier-large', marker_size=3)
    tfb.save_fig_html(fig, format='small', name=f'./fig-effective-xp-multiplier-small', marker_size=3)

## Extras

In [40]:
# Fig. ?: plots XP relative error vs level for encounter with a single monster
import pandas as pd

# pc stats
classes = list(PC_CLASSES.keys())
pc_data_file = f'../../_classes/xp-and-player-characters/pc data - Medium 2 short rests.json'
dfP = load_pc_data(pc_data_file, ['class','level','hit points mean','hit points multiplier','damage per round mean','attack bonus','armor class'])
dfP['adj hit points mean'] = dfP['hit points mean']*dfP['hit points multiplier']
dfP = dfP.rename({'adj hit points mean': 'adj HP', 'damage per round mean': 'adj DPR', 'attack bonus': 'adj AB', 'armor class': 'adj AC'}, axis=1)
xp_values = 0.5*np.array(ENCOUNTER_DIFFICULTIES['Daily']['XP'])
dfP['Encounter XP'] = dfP.apply(lambda row: xp_values[row['level']-1], axis=1)
dfP = dfP[dfP['class'].isin(classes)]
dfP = dfP[['level','adj HP','adj DPR','adj AC','adj AB','Encounter XP']].groupby(['level']).mean()

# monster stats
dfM = pd.read_csv('../../assets/data/monsters-2014.csv')
dfM['adj AC']  = 0.5*(dfM['adj AC'] + dfM['adj SB'] + 14)
dfM = dfM[dfM['CR'].between(1, 28)]
dfM.sort_values(by='CR', inplace=True)
dfM = dfM[['CR','HP','adj HP','AC','adj AC','adj DPR','adj AB','XP']].groupby(['CR']).mean()

# create figure
fig = go.Figure()

colors = iter(COLORS)

########################################################################################
name = f'5e official'
levels = dfP.index.tolist()
rcrs = np.linspace(-28,28,57)
x_all = []
y_all = []
text = []
for lvl in levels:
    for rcr in rcrs:
        cr = rcr+lvl
        if (cr > 0) and (cr <=28):
            dxp = calc_xp_ratio(dfP, dfM, 4*[lvl], cr)
            ddmg = calc_damage_ratio(dfP, dfM, 4*[lvl], cr)
            accuracy = (dxp - ddmg)/ddmg
            x_all.append(rcr)
            y_all.append(accuracy)
            text.append(
                f'<b>{name}</b><br>'+
                f'accuracy: {accuracy:.2f}<br>'+
                f'CR - level: {rcr:.0f}<br>'+
                f'CR: {cr:.0f}<br>'+
                f'level: {lvl:.0f}')

dfA = pd.DataFrame({'rel CR': tfb.jitter(x_all, 0.01), 'accuracy': y_all, 'text': text})
#dfA = dfA[dfA['accuracy'].gt(0)]
dfA.sort_values(by='rel CR', inplace=True)

smooth_factor = 1.e8
fig = tfb.plot_data_and_spline(fig, 
    x=dfA['rel CR'], 
    y=dfA['accuracy'],
    smooth_factor=smooth_factor, 
    name=name,
    legendgroup=name,
    line=dict(color=next(colors), dash='solid', width=2),
    marker_size=3,
    hovertext=dfA['text'],
    hoverinfo='text',
)

########################################################################################
name = f'XP - linear'
xp_method = 'linear'
dfP['Encounter eXP']  = dfP.apply(lambda row: estats5e.effXP(row['adj HP'], row['adj AC'], row['adj DPR'], row['adj AB'], method=xp_method, ctype='PC'), axis=1)
dfM['eXP']  = dfM.apply(lambda row: estats5e.effXP(row['adj HP'], row['adj AC'], row['adj DPR'], row['adj AB'], method=xp_method, ctype='NPC'), axis=1)

levels = dfP.index.tolist()
rcrs = np.linspace(-28,28,57)
x_all = []
y_all = []
text = []
for lvl in levels:
    for rcr in rcrs:
        cr = rcr+lvl
        if (cr > 0) and (cr <=28):
            dxp = calc_exp_ratio(dfP, dfM, 4*[lvl], cr)
            ddmg = calc_damage_ratio(dfP, dfM, 4*[lvl], cr)
            accuracy = (dxp - ddmg)/ddmg
            x_all.append(rcr)
            y_all.append(accuracy)
            text.append(
                f'<b>{name}</b><br>'+
                f'accuracy: {accuracy:.2f}<br>'+
                f'CR - level: {rcr:.0f}<br>'+
                f'CR: {cr:.0f}<br>'+
                f'level: {lvl:.0f}')

dfA = pd.DataFrame({'rel CR': tfb.jitter(x_all, 0.01), 'accuracy': y_all, 'text': text})
#dfA = dfA[dfA['accuracy'].gt(0)]
dfA.sort_values(by='rel CR', inplace=True)

smooth_factor = 1.e8
fig = tfb.plot_data_and_spline(fig, 
    x=dfA['rel CR'], 
    y=dfA['accuracy'],
    smooth_factor=smooth_factor, 
    name=name,
    legendgroup=name,
    line=dict(color=next(colors), dash='solid', width=2),
    marker_size=3,
    hovertext=dfA['text'],
    hoverinfo='text',
)

########################################################################################
name = f'XP - exponential'
xp_method = 'exp'
dfP['Encounter eXP']  = dfP.apply(lambda row: estats5e.effXP(row['adj HP'], row['adj AC'], row['adj DPR'], row['adj AB'], method=xp_method, ctype='PC'), axis=1)
dfM['eXP']  = dfM.apply(lambda row: estats5e.effXP(row['adj HP'], row['adj AC'], row['adj DPR'], row['adj AB'], method=xp_method, ctype='NPC'), axis=1)

levels = dfP.index.tolist()
rcrs = np.linspace(-28,28,57)
x_all = []
y_all = []
text = []
for lvl in levels:
    for rcr in rcrs:
        cr = rcr+lvl
        if (cr > 0) and (cr <=28):
            dxp = calc_exp_ratio(dfP, dfM, 4*[lvl], cr)
            ddmg = calc_damage_ratio(dfP, dfM, 4*[lvl], cr)
            accuracy = (dxp - ddmg)/ddmg
            x_all.append(rcr)
            y_all.append(accuracy)
            text.append(
                f'<b>{name}</b><br>'+
                f'accuracy: {accuracy:.2f}<br>'+
                f'CR - level: {rcr:.0f}<br>'+
                f'CR: {cr:.0f}<br>'+
                f'level: {lvl:.0f}')

dfA = pd.DataFrame({'rel CR': tfb.jitter(x_all, 0.01), 'accuracy': y_all, 'text': text})
#dfA = dfA[dfA['accuracy'].gt(0)]
dfA.sort_values(by='rel CR', inplace=True)

smooth_factor = 1.e8
fig = tfb.plot_data_and_spline(fig, 
    x=dfA['rel CR'], 
    y=dfA['accuracy'],
    smooth_factor=smooth_factor, 
    name=name,
    legendgroup=name,
    line=dict(color=next(colors), dash='solid', width=2),
    marker_size=3,
    hovertext=dfA['text'],
    hoverinfo='text',
)

fig.add_scatter(
    x=[-30,30], 
    y=[0,0],
    mode='lines',
    showlegend=False,
    line=dict(color='black', dash='dash'),
    hoverinfo='skip',
)

# show figure
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    margin=dict(l=50, r=25, b=40, t=40, pad=4),
    font=dict(size=12),
    xaxis=dict(
        title_text='challenge rating - party level',
        range=[-20.8,30.8],
        tick0=0, dtick=5,
        minor=dict(tick0=0, dtick=1),
    ),
    yaxis=dict(
        title_text='relative error',
        range=[-1,1],
        tick0=0, dtick=0.5, tickformat='.0%',
        minor=dict(tick0=0, dtick=0.1),
    ),
    legend=dict(
        yanchor='bottom',  y=0.01,
        xanchor='left', x=0.01,
    ),
    width=600,
    height=500,
)
fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-xp-accuracy-vs-relative-cr-large', style='aspect-ratio:600/550', marker_size=3)
#    tfb.save_fig_html(fig, format='small', name=f'./fig-xp-accuracy-vs-relative-cr-small', style='aspect-ratio:600/550', marker_size=3)

In [42]:
# plots the damage ratio for a party of four PCs against a single monster
import pandas as pd

# pc stats
classes = list(PC_CLASSES.keys())
pc_data_file = f'../../_classes/xp-and-player-characters/pc data - Medium 2 short rests.json'
dfP = load_pc_data(pc_data_file, ['class','level','hit points mean','hit points multiplier','damage per round mean','attack bonus','armor class'])
dfP['adj hit points mean'] = dfP['hit points mean']*dfP['hit points multiplier']
dfP = dfP.rename({'adj hit points mean': 'adj HP', 'damage per round mean': 'adj DPR', 'attack bonus': 'adj AB', 'armor class': 'adj AC'}, axis=1)
xp_values = 0.5*np.array(ENCOUNTER_DIFFICULTIES['Daily']['XP'])
dfP['XP'] = dfP.apply(lambda row: xp_values[row['level']-1], axis=1)
dfP = dfP[dfP['class'].isin(classes)]
dfP = dfP[['level','adj HP','adj DPR','adj AC','adj AB','XP']].groupby(['level']).mean()

# monster stats
dfM = pd.read_csv('../../assets/data/monsters-2014.csv')
dfM['adj AC']  = 0.5*(dfM['adj AC'] + dfM['adj SB'] + 14)
dfM = dfM[dfM['CR'].between(1, 28)]
dfM = dfM[['CR','HP','adj HP','AC','adj AC','adj DPR','adj AB','XP']].groupby(['CR']).mean()
dfM.sort_values(by='CR', inplace=True)


# create figure
fig = go.Figure()

"""fig.add_scatter(
    x=dfP['adj AC'], 
    y=dfP['adj AB'],
    name='PCs',
    mode='lines+markers',
)

fig.add_scatter(
    x=dfM['adj AC'], 
    y=dfM['adj AB'],
    name='NPCs',
    mode='lines+markers',
)"""

colors = iter(COLORS)
fig = tfb.plot_data_and_fit(fig, x=dfP['adj AC'], y=dfP['adj AB'], name='PCs', legendgroup='PCs', line_color=next(colors), print_coefficients=True)
fig = tfb.plot_data_and_fit(fig, x=dfM['adj AC'], y=dfM['adj AB'], name='NPCs', legendgroup='NPCs', line_color=next(colors), print_coefficients=True)

# show figure
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(
        title_text='armor class',
        #range=[0.8, 28.2],
        tick0=0, dtick=2,
        minor=dict(tick0=0, dtick=1),
    ),
    yaxis=dict(
        title_text='attack bonus',
        #range=[0, 2],
        tick0=0, dtick=2,
        minor=dict(tick0=0, dtick=2),
        tickformat='.0f',
    ),
    legend=dict(
        yanchor='top',  y=0.99,
        xanchor='left', x=0.01,
    ),
    width=600, 
    height=500,
)
fig.show(config=tfb.FIG_CONFIG)

CR >= 1: -21.76 + 1.78*CR
CR >= 1: -13.91 + 1.29*CR


In [60]:
# plots the damage ratio for a party of four PCs against a single monster
import pandas as pd

# pc stats
classes = list(PC_CLASSES.keys())
pc_data_file = f'../../_classes/xp-and-player-characters/pc data - Medium 2 short rests.json'
dfP = load_pc_data(pc_data_file, ['class','level','hit points mean','hit points multiplier','damage per round mean','attack bonus','armor class'])
dfP['adj hit points mean'] = dfP['hit points mean']*dfP['hit points multiplier']
dfP = dfP.rename({'adj hit points mean': 'adj HP', 'damage per round mean': 'adj DPR', 'attack bonus': 'adj AB', 'armor class': 'adj AC'}, axis=1)
xp_values = 0.5*np.array(ENCOUNTER_DIFFICULTIES['Daily']['XP'])
dfP['Encounter XP'] = dfP.apply(lambda row: xp_values[row['level']-1], axis=1)
dfP = dfP[dfP['class'].isin(classes)]
dfP = dfP[['level','adj HP','adj DPR','adj AC','adj AB','Encounter XP']].groupby(['level']).mean()

# monster stats
dfM = pd.read_csv('../../assets/data/monsters-2014.csv')
dfM = dfM[dfM['CR'].between(1, 30)]
dfM['adj AC']  = 0.5*(dfM['adj AC'] + dfM['adj SB'] + 14)
dfM = dfM[['CR','HP','adj HP','AC','adj AC','adj DPR','adj AB','XP']].groupby(['CR']).mean()
dfM.sort_values(by='CR', inplace=True)


# create figure
fig = go.Figure()

levels = list(range(1,21))
dmg_ratio = [calc_damage_ratio(dfP, dfM, 4*[i], i) for i in levels]
xp_ratio = [calc_xp_ratio(dfP, dfM, 4*[i], i) for i in levels]

tfb.plot_data_and_fit(
    fig, 
    levels, dmg_ratio,
    name='damage',
    legendgroup='damage',
    line=dict(color=COLORS[0], dash='solid', width=1),
    hovertemplate=
            '<b>Monsters</b><br>'+
            'level %{x}<br>'+
            'rounds %{y:.2f}'+
            '<extra></extra>',
    print_coefficients=True,
)

tfb.plot_data_and_fit(
    fig, 
    levels, xp_ratio,
    name='XP',
    legendgroup='XP',
    line=dict(color=COLORS[1], dash='solid', width=1),
    hovertemplate=
            '<b>Monsters</b><br>'+
            'level %{x}<br>'+
            'rounds %{y:.2f}'+
            '<extra></extra>',
    print_coefficients=True,
)

# show figure
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(
        title_text='level',
        range=[0.8, 20.2],
        tick0=0, dtick=5,
        minor=dict(tick0=0, dtick=1),
    ),
    yaxis=dict(
        title_text='damage ratio',
        range=[0, 1],
        tick0=0, dtick=0.2,
        minor=dict(tick0=0, dtick=0.1),
    ),
    legend=dict(
        yanchor='top',  y=0.99,
        xanchor='left', x=0.01,
    ),
    width=600, 
    height=500,
)
fig.show(config=tfb.FIG_CONFIG)

CR >= 1: 0.28 + 0.01*CR
CR >= 1: 0.31 + 0.00*CR


In [62]:
# plots the damage ratio for a party of four PCs against a single monster
import pandas as pd

# pc stats
classes = list(PC_CLASSES.keys())
pc_data_file = f'../../_classes/xp-and-player-characters/pc data - Medium 2 short rests.json'
dfP = load_pc_data(pc_data_file, ['class','level','hit points mean','hit points multiplier','damage per round mean','attack bonus','armor class'])
dfP['adj hit points mean'] = dfP['hit points mean']*dfP['hit points multiplier']
dfP = dfP.rename({'adj hit points mean': 'adj HP', 'damage per round mean': 'adj DPR', 'attack bonus': 'adj AB', 'armor class': 'adj AC'}, axis=1)
xp_values = 0.5*np.array(ENCOUNTER_DIFFICULTIES['Daily']['XP'])
dfP['Encounter XP'] = dfP.apply(lambda row: xp_values[row['level']-1], axis=1)
dfP = dfP[dfP['class'].isin(classes)]
dfP = dfP[['level','adj HP','adj DPR','adj AC','adj AB','Encounter XP']].groupby(['level']).mean()

# monster stats
dfM = pd.read_csv('../../assets/data/monsters-2014.csv')
dfM['adj AC']  = 0.5*(dfM['adj AC'] + dfM['adj SB'] + 14)
dfM = dfM[dfM['CR'].between(1, 28)]
dfM = dfM[['CR','HP','adj HP','AC','adj AC','adj DPR','adj AB','XP']].groupby(['CR']).mean()
dfM.sort_values(by='CR', inplace=True)


# create figure
fig = go.Figure()

levels = [5,11,17]
crs = dfM.index.tolist()
colors = iter(COLORS)
for level in levels:
    dmg_ratio = [calc_damage_ratio(dfP, dfM, 4*[level], int(cr)) for cr in crs]
    xp_ratio = [calc_xp_ratio(dfP, dfM, 4*[level], int(cr)) for cr in crs]
    color = next(colors)
    
    tfb.plot_data_and_fit(
        fig, 
        crs, np.array(dmg_ratio)/np.array(xp_ratio),
        name=f'level {level}',
        legendgroup=f'level {level}',
        line=dict(color=color, dash='solid', width=1),
        hovertemplate=
                '<b>Monsters</b><br>'+
                'level %{x}<br>'+
                'rounds %{y:.2f}'+
                '<extra></extra>',
        print_coefficients=True,
    )

fig.add_scatter(
    x=crs, 
    y=[1]*len(crs),
    showlegend=False,
    mode='lines',
    line_color='black',
    line_dash='dash',
)

# show figure
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(
        title_text='challenge rating',
        range=[0.8, 28.2],
        tick0=0, dtick=5,
        minor=dict(tick0=0, dtick=1),
    ),
    yaxis=dict(
        title_text='damage ratio / XP ratio',
        range=[0, 2],
        tick0=0, dtick=0.2,
        minor=dict(tick0=0, dtick=0.1),
        tickformat='.1f',
    ),
    legend=dict(
        yanchor='top',  y=0.99,
        xanchor='left', x=0.01,
    ),
    width=600, 
    height=500,
)
fig.show(config=tfb.FIG_CONFIG)

CR >= 1: 0.49 + 0.10*CR
CR >= 1: 0.42 + 0.05*CR
CR >= 1: 0.38 + 0.04*CR
